# S5.G1-2 · G3 五臂对比 Santander：联邦臂 A（隐语 SGB）

内地为主动方（持标签）并持有私钥，香港为被动方，只持公钥做同态计算（OU）；单进程两方模拟（`debug_mode`），耗时只作相对比较（DR-GOV-016 · D-3）。
超参取自 S5.G1-1 在集中式明文代理 L4 上按内层验证集选定的配置（冻结修改 A-4）。打分只写入私有目录。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_g1_santander_run.yaml"
run = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
PRE_PATH = REPO_ROOT / run["preregistration"]
pre = yaml.safe_load(PRE_PATH.read_text(encoding="utf-8"))
fz = pre["freeze"]
ad = fz["arm_details"]


def sha(path):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:HASH_PREFIX_LEN]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("preregistration:", PRE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(PRE_PATH), "| status:", pre["status"])
print("seed:", run["seed"], "| seeds:", run["seeds"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
print("step:", run["step_id"], "|", run["purpose"])
mismatch = [p for p, h in fz["code_sha256"].items() if hashlib.sha256((REPO_ROOT / p).read_bytes()).hexdigest() != h]
print("冻结代码指纹核对：", "一致" if not mismatch else "不一致 " + str(mismatch))
assert pre["status"] == "frozen" and not mismatch
PRIV = pathlib.Path(os.environ[run["private_root_env"]])
SEED0 = run["seeds"][0]
MB, NJ = ad["xgboost_max_bin"], ad["n_jobs"]

config: modules/m5_modeling/configs/s5_g1_santander_run.yaml sha256:09834fbb00ea
preregistration: modules/m6_evaluation/configs/s6_p3_g3_preregistration.yaml sha256:c0a4bbfc9709 | status: frozen
seed: 11 | seeds: [11, 22, 33, 44, 55, 66]
git: f80f74f（工作区含本步未提交改动）
step: S5.G1 | 按冻结的 S6.P3 在 Santander 上训练 L0、L1（k=5/10/20）、B_T1、B_T2、A（SGB）、集中式代理 L4 与阴性对照 NC，并做配对比较
冻结代码指纹核对： 一致


In [2]:
from modules.m5_modeling.components import g3_sgb as SG

arr = np.load(PRIV / run["arrays_file"])
TUNING = json.loads((PRIV / run["tuning_file"]).read_text(encoding="utf-8"))
params = SG.sgb_params(TUNING["A"]["best"], ad["sgb_sketch_eps"], SEED0)
print("A 的 SGB 参数（来自 L4 集中式代理调参）：", params)
sets = {n: (arr[n + "__xg"], arr[n + "__xh"]) for n in ("inner_val", "dev_eval")}
scores, timing = SG.fit_and_score(arr["train__xg"], arr["train__xh"], arr["train__y"], sets, params, ad["sgb_he_schema"])
np.savez_compressed(PRIV / run["sgb_scores_file"], **scores)
from sklearn.metrics import roc_auc_score
print("SGB 训练 %.0f 秒，打分 %.0f 秒" % (timing["fit_seconds"], timing["predict_seconds"]))
print("内层验证集 AUROC：SGB %.4f ｜ L4 集中式代理调参时 %.4f"
      % (roc_auc_score(arr["inner_val__y"], scores["inner_val"]), TUNING["L4"]["best_val_auc"]))
(PRIV / "g3_santander_sgb_timing.json").write_text(json.dumps(timing), encoding="utf-8")

A 的 SGB 参数（来自 L4 集中式代理调参）： {'num_boost_round': 150, 'max_depth': 5, 'learning_rate': 0.1, 'reg_lambda': 10.0, 'sketch_eps': 0.03, 'objective': 'logistic', 'seed': 11}


[已折叠 150 行相同的 SecretFlow 运行警告：WARNING:root:no effective dump_function provided.]


[2026-10-26 12:05:38.830] [info] [bigint_spi.cc:79] The default library used for BigInt operations is openssl


[2026-10-26 12:05:42.188] [info] [thread_pool.cc:30] Create a fixed thread pool with size 1


SGB 训练 3325 秒，打分 4 秒
内层验证集 AUROC：SGB 0.8829 ｜ L4 集中式代理调参时 0.8836


72

**读法**：

- SGB 用 L4 集中式代理选出的配置（150 棵、深 5、学习率 0.1、L2 正则 10、分桶 sketch_eps 0.03）训练一次，单进程两方模拟耗时约 55 分钟（2 核，只作相对比较；L4 明文约 1 秒）。
- 内层验证集 AUROC：SGB 0.8829，L4 0.8836，相差 0.0007，与 SecureBoost 类协议无损的性质一致；开发评估集上的正式核对见 S5.G1-3 的「A − L4」。
- 上方的 SecretFlow 运行警告（no effective dump_function provided）每棵树一行，已折叠为一行说明；没有改动任何其他输出。